# SIC 2026 · Inteligencia Artificial
## Clase 18: Fine-tuning, LoRA, cuantización y sentimiento por aspecto (Capítulo 10, Unidades 2 a 4)

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Es la última clase de contenidos antes del capstone. En la Clase 17 usamos modelos preentrenados **tal como vienen**. Hoy los **ajustamos** a una tarea propia, los hacemos más livianos y los aplicamos a un problema real de negocio.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 18:00 a 18:25 | 1. Qué es el fine-tuning | 40 a 51 | La idea y el flujo de trabajo |
| 18:25 a 19:35 | 2. Práctica de fine-tuning | 52 a 73 | Ajustar BETO con reseñas de Amazon en español y compararlo con todo lo anterior |
| 19:35 a 19:45 | Pausa | | |
| 19:45 a 20:25 | 3. LoRA | 74 a 82 | El mismo ajuste, entrenando menos del 1% de los parámetros |
| 20:25 a 21:00 | 4. Cuantización | 83 a 103 | Hacer el modelo más liviano y rápido |
| 21:00 a 21:50 | 5. Sentimiento por aspecto (ABSA) | 104 a 130 | ¿Qué opina el cliente de la batería? ¿Y de la pantalla? |
| 21:50 a 22:00 | 6. Cierre del curso | | Un mapa de todo lo visto, para el capstone |

**Antes de empezar:** *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*. Sin GPU, los entrenamientos de las Partes 2, 3 y 5 tardan demasiado.

Hay 12 ejercicios; las soluciones están al final.

In [ ]:
!pip install -q transformers datasets peft accelerate

import os, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

GPU = torch.cuda.is_available()
DISPOSITIVO = 0 if GPU else -1
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print("GPU:", torch.cuda.get_device_name(0) if GPU else "no (activa la GPU en Entorno de ejecución)")

---
# Parte 1. Qué es el fine-tuning (láminas 40 a 51)

> **Fine-tuning (ajuste fino):** tomar un modelo **preentrenado** con enormes cantidades de texto general y **seguir entrenándolo** con pocos datos de nuestra tarea, para especializarlo.

Es como contratar a alguien que ya sabe leer y escribir muy bien, y enseñarle en una semana el vocabulario de nuestra empresa. Partir de cero sería enseñarle a leer.

## Paso 1.1. Las tres formas de usar un modelo de lenguaje

| | Entrenar desde cero | Usar sin ajustar | **Fine-tuning** |
|---|---|---|---|
| Lo hicimos en | Clases 13 a 16 (LSTM) | Clase 17 (pipeline) | **Hoy** |
| Datos propios que necesita | Muchísimos | Ninguno | Cientos o miles |
| Se adapta a nuestra tarea | Sí, pero sabe poco del idioma | No: hace la tarea para la que lo entrenaron otros | **Sí, y conoce el idioma** |
| Costo | Alto | Nulo | Moderado (minutos con GPU) |

## Paso 1.2. ¿Qué se ajusta? La "cabeza" del modelo

Un modelo como BERT tiene dos partes:

| Parte | Qué hace | En el fine-tuning |
|---|---|---|
| **Cuerpo** (12 capas de transformer) | Entiende el texto: convierte cada token en un vector que depende del contexto | Parte con lo aprendido y se ajusta un poco |
| **Cabeza** (una capa densa) | Convierte esa comprensión en la respuesta de **nuestra** tarea | Es **nueva**: parte al azar y se entrena desde cero |

Cuando escribimos `AutoModelForSequenceClassification.from_pretrained(modelo, num_labels=2)`, Hugging Face carga el cuerpo entrenado y le pone una cabeza nueva de 2 salidas. Por eso aparece un aviso de que algunos pesos "no fueron inicializados": es la cabeza.

## Paso 1.3. El flujo de trabajo (láminas 45 a 51)

| Paso | Qué se hace | En código |
|---|---|---|
| 1. Datos | Textos con sus etiquetas, separados en entrenamiento, validación y prueba | `pandas`, `Dataset` |
| 2. Tokenizar | Con **el tokenizador del modelo** que vamos a ajustar | `tokenizer(...)` |
| 3. Modelo | Cuerpo preentrenado y cabeza nueva | `AutoModelForSequenceClassification` |
| 4. Entrenar | Hugging Face trae su propio ciclo, como el `fit` de Keras | `Trainer` y `TrainingArguments` |
| 5. Evaluar | En datos que el modelo no vio | `trainer.predict` |
| 6. Guardar y usar | Guardar el modelo y cargarlo como pipeline | `save_pretrained` |

**La tasa de aprendizaje** es mucho más baja que en las Clases 14 a 16 (2 × 10⁻⁵ en vez de 10⁻³): el cuerpo ya sabe mucho, y queremos **ajustarlo**, no borrarlo.

---
# Parte 2. Práctica de fine-tuning (láminas 52 a 73)

## Paso 2.1. Los datos: reseñas de Amazon en español

Usamos reseñas de productos de Amazon **en español**, cada una con su nota de 1 a 5 estrellas. Las convertimos en un problema binario, como en las Clases 11 y 13: **negativa** (1 o 2 estrellas) o **positiva** (4 o 5), y dejamos fuera las de 3.

Para que el entrenamiento dure pocos minutos, usamos **4.000 reseñas** de entrenamiento, 1.000 de validación y 1.000 de prueba, balanceadas.

**Si el dataset en español no está disponible**, el notebook usa automáticamente IMDb en inglés y un modelo en inglés. Todo lo demás funciona igual.

In [ ]:
def a_binario(df, col_texto, col_estrellas, base_estrellas):
    """Deja texto y etiqueta 0 (1-2 estrellas) o 1 (4-5 estrellas)."""
    estrellas = df[col_estrellas] + (1 - base_estrellas)        # pasa a escala 1 a 5
    df = df.assign(estrellas=estrellas)
    df = df[df["estrellas"] != 3]
    return pd.DataFrame({"text": df[col_texto].values, "label": (df["estrellas"] >= 4).astype(int).values})

def tomar(df, n, semilla):
    """n ejemplos balanceados: la mitad de cada clase."""
    partes = [g.sample(n // 2, random_state=semilla) for _, g in df.groupby("label")]
    return pd.concat(partes).sample(frac=1, random_state=semilla).reset_index(drop=True)

def cargar_datos():
    for nombre, config in [("mteb/amazon_reviews_multi", "es"), ("SetFit/amazon_reviews_multi_es", None)]:
        try:
            ds = load_dataset(nombre, config) if config else load_dataset(nombre)
            tr, te = ds["train"].to_pandas(), ds["test"].to_pandas()
            col_t = "text" if "text" in tr else "review_body"
            col_e = "label" if "label" in tr else "stars"
            base = 0 if tr[col_e].min() == 0 else 1
            tr, te = a_binario(tr, col_t, col_e, base), a_binario(te, col_t, col_e, base)
            print("Datos:", nombre, "(español)")
            return tr, te, "es"
        except Exception as e:
            print("No se pudo cargar", nombre, "|", type(e).__name__)
    imdb = load_dataset("stanfordnlp/imdb")
    print("Datos: IMDb (inglés)")
    return imdb["train"].to_pandas(), imdb["test"].to_pandas(), "en"

completo_tr, completo_te, IDIOMA = cargar_datos()
entreno = tomar(completo_tr, 4000, 1)
resto = completo_tr[~completo_tr["text"].isin(entreno["text"])]
validacion = tomar(resto, 1000, 2)
prueba = tomar(completo_te, 1000, 3)
print("Entrenamiento:", len(entreno), "| Validación:", len(validacion), "| Prueba:", len(prueba))
entreno.head()

**Qué deberías ver:** de dónde salieron los datos, y 4.000, 1.000 y 1.000 reseñas con su etiqueta 0 o 1.

### Ejercicio 1
Comprueba que el entrenamiento está **balanceado**: guarda en `prop_positivas` la proporción de reseñas positivas en `entreno`.

In [ ]:
prop_positivas = None  # TU CÓDIGO AQUÍ
print(prop_positivas)

In [ ]:
revisar(lambda: np.isclose(prop_positivas, 0.5))

## Paso 2.2. Las líneas base: lo que ya sabíamos hacer

Antes de ajustar nada, medimos dos cosas en las mismas 1.000 reseñas de prueba:

1. **TF-IDF y regresión logística** (Clase 11), entrenada con las mismas 4.000 reseñas.
2. **Un modelo preentrenado sin ajustar** (Clase 17): el de 1 a 5 estrellas, contando como positiva una predicción de 4 o 5 estrellas.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

resultados = {}
tfidf_log = make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2)), LogisticRegression(max_iter=2000, C=3))
tfidf_log.fit(entreno["text"], entreno["label"])
resultados["TF-IDF + logística (Clase 11)"] = accuracy_score(prueba["label"], tfidf_log.predict(prueba["text"]))
print(resultados)

### Ejercicio 2
Usa el pipeline de estrellas de la Clase 17 sobre las reseñas de prueba y guarda su exactitud en `exac_sin_ajustar`. Cuenta como positiva (1) una predicción de **4 o 5** estrellas y como negativa (0) cualquier otra. Pista: la etiqueta viene como `"4 stars"`; su primer carácter es la nota.

In [ ]:
estrellas = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment",
                     device=DISPOSITIVO, truncation=True)
salidas_estrellas = estrellas(prueba["text"].tolist(), batch_size=64)

exac_sin_ajustar = None  # TU CÓDIGO AQUÍ
resultados["Preentrenado sin ajustar (Clase 17)"] = exac_sin_ajustar
print(exac_sin_ajustar)

In [ ]:
revisar(lambda: np.isclose(exac_sin_ajustar, accuracy_score(prueba["label"], [1 if int(s["label"][0]) >= 4 else 0 for s in salidas_estrellas])))

## Paso 2.3. Tokenizar

Usamos **BETO**, el BERT en español de la Universidad de Chile que conocimos en la Clase 17 (o un DistilBERT en inglés si los datos son de IMDb). Cada reseña se corta o rellena a 128 tokens: es el relleno de la Clase 12, ahora con el tokenizador del modelo.

In [ ]:
MODELO_BASE = "dccuchile/bert-base-spanish-wwm-uncased" if IDIOMA == "es" else "distilbert-base-uncased"
tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)

def tokenizar(lote):
    return tokenizador(lote["text"], padding="max_length", truncation=True, max_length=128)

ds_tr = Dataset.from_pandas(entreno).map(tokenizar, batched=True)
ds_va = Dataset.from_pandas(validacion).map(tokenizar, batched=True)
ds_te = Dataset.from_pandas(prueba).map(tokenizar, batched=True)

ejemplo = ds_tr[0]
print("Columnas:", ds_tr.column_names)
print("Primeros 20 tokens:", tokenizador.convert_ids_to_tokens(ejemplo["input_ids"][:20]))
print("Máscara de atención (1 = token real, 0 = relleno):", ejemplo["attention_mask"][:20], "...")

**Qué deberías ver:** las columnas nuevas `input_ids` (los números de los tokens) y `attention_mask` (que le dice al modelo qué tokens son reales y cuáles son relleno, para que los ignore).

## Paso 2.4. El modelo y el entrenamiento

`TrainingArguments` son los argumentos del `fit` de Keras, con otros nombres:

| Argumento | Equivale en Keras a | Valor |
|---|---|---|
| `num_train_epochs` | `epochs` | 2 |
| `per_device_train_batch_size` | `batch_size` | 16 |
| `learning_rate` | la tasa de Adam | 2 × 10⁻⁵ |
| `eval_strategy="epoch"` | `validation_data` | Evalúa al final de cada época |
| `fp16=True` | | Usa números de 16 bits en la GPU: casi el doble de rápido |

In [ ]:
def metricas(eval_pred):
    logits, etiquetas = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"exactitud": accuracy_score(etiquetas, pred), "f1": f1_score(etiquetas, pred)}

def argumentos(carpeta, tasa, epocas=2):
    return TrainingArguments(output_dir=carpeta, eval_strategy="epoch", save_strategy="no",
                             learning_rate=tasa, num_train_epochs=epocas,
                             per_device_train_batch_size=16, per_device_eval_batch_size=64,
                             weight_decay=0.01, fp16=GPU, logging_steps=50, report_to="none", seed=42)

ETIQUETAS = dict(num_labels=2, id2label={0: "negativa", 1: "positiva"}, label2id={"negativa": 0, "positiva": 1})
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, **ETIQUETAS)
total_params = sum(p.numel() for p in modelo.parameters())
print(f"Parámetros del modelo: {total_params:,}")

trainer = Trainer(model=modelo, args=argumentos("./resultados_ft", 2e-5),
                  train_dataset=ds_tr, eval_dataset=ds_va, compute_metrics=metricas)
inicio = time.time()
trainer.train()
tiempo_completo = time.time() - inicio
print(f"\nTiempo de entrenamiento: {tiempo_completo:.0f} segundos")

**Qué deberías ver:** cerca de **110 millones** de parámetros con BETO, una tabla con la pérdida y la exactitud de validación por época, y unos 3 a 5 minutos de entrenamiento con GPU.

## Paso 2.5. Evaluar en prueba (lámina 68)

In [ ]:
from sklearn.metrics import confusion_matrix

salida_prueba = trainer.predict(ds_te)
pred_ft = salida_prueba.predictions.argmax(axis=-1)
print("Matriz de confusión:\n", confusion_matrix(prueba["label"], pred_ft))

### Ejercicio 3
Guarda en `exac_ft` la exactitud del modelo ajustado en prueba, agrégala a `resultados` con el nombre `"Fine-tuning completo"` y muestra la tabla ordenada.

In [ ]:
exac_ft = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.isclose(exac_ft, accuracy_score(prueba["label"], pred_ft)) and "Fine-tuning completo" in resultados)

**Qué deberías ver:** el modelo ajustado arriba de todo, cerca de **0,92 o 0,93** con las reseñas en español; TF-IDF cerca de 0,87; y el preentrenado sin ajustar en un punto intermedio. Con solo 4.000 ejemplos y unos minutos de GPU, el ajuste fino supera a todo lo que habíamos hecho.

## Paso 2.6. Guardar y volver a cargar (láminas 70 y 71)

Un modelo ajustado se guarda en una carpeta (pesos, configuración y tokenizador) y después se carga como cualquier modelo de Hugging Face. Esa carpeta también se puede subir al Hub o usar en una app de Gradio (Clase 6).

In [ ]:
CARPETA = "mi_modelo_resenas"
trainer.save_model(CARPETA)
tokenizador.save_pretrained(CARPETA)
print("Archivos:", os.listdir(CARPETA))

mi_clasificador = pipeline("text-classification", model=CARPETA, device=DISPOSITIVO)
frases = ["Llegó rápido y funciona de maravilla, lo recomiendo.",
          "Se rompió a la semana y no me quisieron devolver el dinero.",
          "No es lo que esperaba, pero cumple."]
for f, r in zip(frases, mi_clasificador(frases)):
    print(f"{r['label']:9s} {r['score']:.2f} | {f}")

### Ejercicio 4
Escribe una reseña propia en `mi_resena` (en el idioma de los datos) y guarda en `etiqueta_mi_resena` la etiqueta que le asigna `mi_clasificador`.

In [ ]:
mi_resena = ""             # TU RESEÑA AQUÍ
etiqueta_mi_resena = None  # TU CÓDIGO AQUÍ
print(mi_resena, "→", etiqueta_mi_resena)

In [ ]:
revisar(lambda: len(mi_resena.split()) >= 3 and etiqueta_mi_resena == mi_clasificador(mi_resena)[0]["label"])

---
# Parte 3. LoRA: ajustar entrenando muy pocos parámetros (láminas 74 a 82)

En la Parte 2 ajustamos **todos** los parámetros. Con modelos grandes (miles de millones de parámetros, como los LLM) eso es imposible en Colab: no cabe en memoria. **PEFT** (*Parameter-Efficient Fine-Tuning*) es una familia de técnicas para ajustar solo una pequeña parte.

## Paso 3.1. La idea de LoRA (lámina 76)

Cada capa del modelo tiene matrices de pesos grandes, por ejemplo **W** de 768 × 768. Ajustarla completa es cambiar 589.824 números.

**LoRA** (*Low-Rank Adaptation*) deja **W congelada** y aprende el cambio como el producto de **dos matrices delgadas**:

$$W_{\text{nuevo}} = W + B \cdot A$$

| Matriz | Forma | Parámetros con r = 8 |
|---|---|---|
| W (congelada) | 768 × 768 | 589.824, no se entrenan |
| A | 8 × 768 | 6.144 |
| B | 768 × 8 | 6.144 |
| **B · A** | 768 × 768 | **12.288 entrenables: un 2% de W** |

El número **r** (el rango) controla cuántos parámetros se agregan: más grande, más capacidad y más parámetros.

| Parámetro de `LoraConfig` | Qué controla | Valor típico |
|---|---|---|
| `r` | El rango: el "ancho" de A y B | 8 o 16 |
| `lora_alpha` | Cuánto pesa el cambio B · A | 2 a 4 veces `r` |
| `target_modules` | A qué matrices se agrega LoRA | En BERT: `query` y `value` de la atención |
| `lora_dropout` | Dropout dentro de LoRA (Clase 14) | 0,1 |

### Ejercicio 5
Calcula los parámetros que agrega LoRA a **una** matriz de 768 × 768 con **r = 16**. Guárdalo en `params_lora_r16`.

In [ ]:
params_lora_r16 = None  # TU CÓDIGO AQUÍ
print(params_lora_r16)

In [ ]:
revisar(lambda: params_lora_r16 == 16 * 768 + 768 * 16)

## Paso 3.2. Ajustar con LoRA (láminas 79 y 80)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

modelo_base_lora = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, **ETIQUETAS)
config_lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=32, lora_dropout=0.1,
                         target_modules=["query", "value"] if IDIOMA == "es" else ["q_lin", "v_lin"])
modelo_lora = get_peft_model(modelo_base_lora, config_lora)
modelo_lora.print_trainable_parameters()

entrenables = sum(p.numel() for p in modelo_lora.parameters() if p.requires_grad)
print(f"Entrenables: {entrenables:,}")

**Qué deberías ver:** menos del **1%** de los parámetros como entrenables: las matrices A y B de cada capa, más la cabeza de clasificación.

Con LoRA se usa una tasa de aprendizaje **más alta** (5 × 10⁻⁴), porque los parámetros nuevos son pocos y uno de ellos (B) parte en cero.

In [ ]:
trainer_lora = Trainer(model=modelo_lora, args=argumentos("./resultados_lora", 5e-4),
                       train_dataset=ds_tr, eval_dataset=ds_va, compute_metrics=metricas)
inicio = time.time()
trainer_lora.train()
tiempo_lora = time.time() - inicio

exac_lora = accuracy_score(prueba["label"], trainer_lora.predict(ds_te).predictions.argmax(axis=-1))
resultados["Fine-tuning con LoRA"] = exac_lora
print(f"\nLoRA: exactitud {exac_lora:.3f} en {tiempo_lora:.0f} s | Completo: {exac_ft:.3f} en {tiempo_completo:.0f} s")

### Ejercicio 6
Guarda en `pct_entrenable` el **porcentaje** de parámetros entrenables con LoRA (entrenables / total del modelo × 100), usando `entrenables` y `total_params`.

In [ ]:
pct_entrenable = None  # TU CÓDIGO AQUÍ
print(pct_entrenable)

In [ ]:
revisar(lambda: np.isclose(pct_entrenable, entrenables / total_params * 100, rtol=1e-3) and pct_entrenable < 2)

**Qué deberías ver:** con LoRA, una exactitud **muy cercana** a la del ajuste completo (uno o dos puntos menos), entrenando cerca del 0,3% de los parámetros. Con BERT el ahorro de tiempo es moderado; la gran diferencia aparece con modelos de miles de millones de parámetros, donde LoRA es lo que permite ajustarlos en una sola GPU. Así se ajustan hoy los LLM.

Otra ventaja: el ajuste de LoRA se guarda como un archivo **pequeño** (unos pocos MB, solo A y B), que se combina con el modelo base. Una empresa puede tener un modelo base y decenas de "adaptadores" LoRA, uno por tarea.

---
# Parte 4. Cuantización (láminas 83 a 103)

Un modelo ajustado funciona muy bien, pero pesa cientos de MB y en un computador sin GPU puede ser lento. La **optimización** busca hacerlo más liviano y rápido sin perder mucha exactitud (láminas 83 a 93):

| Técnica | Idea |
|---|---|
| **Poda** | Eliminar los pesos o neuronas que casi no aportan |
| **Destilación** | Entrenar un modelo pequeño (el estudiante) para imitar a uno grande (el profesor). DistilBERT es eso: un BERT destilado |
| **Cuantización** | Guardar cada peso con **menos bits** |

## Paso 4.1. La idea de la cuantización

Cada peso del modelo se guarda como un número de **32 bits** (FP32). La cuantización lo guarda con **8 bits** (INT8): solo 256 valores posibles en vez de 4 mil millones. Es como redondear: se pierde algo de precisión, pero el modelo ocupa menos y calcula más rápido.

Usamos **cuantización dinámica** de PyTorch, como en la lámina 98: convierte a INT8 todas las capas `Linear` (las capas densas). Funciona en **CPU**, que es justamente donde más se necesita.

In [ ]:
import copy
from torch.ao.quantization import quantize_dynamic

modelo_fp32 = copy.deepcopy(trainer.model).cpu().float().eval()
modelo_int8 = quantize_dynamic(modelo_fp32, {torch.nn.Linear}, dtype=torch.qint8)

def tamano_mb(m, nombre):
    torch.save(m.state_dict(), nombre)
    mb = os.path.getsize(nombre) / 1e6
    os.remove(nombre)
    return mb

mb_fp32, mb_int8 = tamano_mb(modelo_fp32, "fp32.pt"), tamano_mb(modelo_int8, "int8.pt")
print(f"FP32: {mb_fp32:.0f} MB | INT8: {mb_int8:.0f} MB")

**Qué deberías ver:** con BETO, el modelo baja de cerca de **440 MB** a cerca de **180 MB** (los embeddings no se cuantizan, por eso no llega a un cuarto).

### Ejercicio 7
Guarda en `reduccion_pct` el porcentaje de reducción de tamaño: (FP32 − INT8) / FP32 × 100.

In [ ]:
reduccion_pct = None  # TU CÓDIGO AQUÍ
print(reduccion_pct)

In [ ]:
revisar(lambda: np.isclose(reduccion_pct, (mb_fp32 - mb_int8) / mb_fp32 * 100))

## Paso 4.2. ¿Pierde exactitud? ¿Gana velocidad?

Comparamos los dos modelos **en CPU** con 300 reseñas de prueba.

In [ ]:
muestra = prueba.sample(300, random_state=0)
entradas = tokenizador(muestra["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")

def medir(m):
    inicio = time.time()
    preds = []
    with torch.no_grad():
        for i in range(0, len(muestra), 32):
            lote = {k: v[i:i + 32] for k, v in entradas.items()}
            preds.extend(m(**lote).logits.argmax(-1).tolist())
    return accuracy_score(muestra["label"], preds), time.time() - inicio

exac_32, t_32 = medir(modelo_fp32)
exac_8, t_8 = medir(modelo_int8)
tabla_cuant = pd.DataFrame({"FP32 (original)": [mb_fp32, exac_32, t_32], "INT8 (cuantizado)": [mb_int8, exac_8, t_8]},
                           index=["Tamaño (MB)", "Exactitud", "Segundos en CPU (300 reseñas)"]).round(3)
tabla_cuant

**Qué deberías ver:** el modelo cuantizado **más chico y más rápido** (cerca de 1,5 a 2 veces), con una exactitud **casi igual** (a veces idéntica, a veces uno o dos aciertos menos).

### Ejercicio 8
Guarda en `aceleracion` cuántas veces más rápido es el modelo INT8 (tiempo FP32 / tiempo INT8), y en `perdida_exactitud` cuánta exactitud perdió (FP32 − INT8).

In [ ]:
aceleracion = None        # TU CÓDIGO AQUÍ
perdida_exactitud = None  # TU CÓDIGO AQUÍ
print(aceleracion, perdida_exactitud)

In [ ]:
revisar(lambda: np.isclose(aceleracion, t_32 / t_8) and np.isclose(perdida_exactitud, exac_32 - exac_8))

**¿Cuándo importa?** Cuando el modelo tiene que funcionar **sin GPU**: en un servidor barato, en un Space gratuito de Hugging Face (Clase 6) o en un dispositivo. Los LLM que se usan en computadores personales casi siempre están cuantizados a 8 o incluso 4 bits.

---
# Parte 5. Sentimiento por aspecto, ABSA (láminas 104 a 130)

## Paso 5.1. El problema (láminas 111 y 112)

*"La pantalla se ve increíble, lástima que la batería no alcanza para el día."*

¿Es positiva o negativa? **Las dos cosas**: positiva sobre la pantalla y negativa sobre la batería. Un clasificador de sentimiento general (Parte 2) tiene que elegir una sola. El **análisis de sentimiento basado en aspectos** (ABSA) responde por separado para cada aspecto del producto, que es lo que necesita un equipo de producto: saber **qué** gusta y **qué** molesta.

## Paso 5.2. El truco: una entrada por aspecto (láminas 113 y 114)

La misma reseña se convierte en **una fila por aspecto**, marcando el aspecto al comienzo:

| Texto de entrada | Etiqueta |
|---|---|
| `[ASPECTO] pantalla [SEP] La pantalla se ve increíble, lástima que la batería...` | positiva |
| `[ASPECTO] batería [SEP] La pantalla se ve increíble, lástima que la batería...` | negativa |

Así, el mismo clasificador binario de la Parte 2 aprende a **mirar el aspecto indicado**.

### Ejercicio 9
Completa `formato_absa(aspecto, texto)` para que devuelva el texto en el formato de la tabla: `"[ASPECTO] "`, el aspecto, `" [SEP] "` y el texto.

In [ ]:
def formato_absa(aspecto, texto):
    return None  # TU CÓDIGO AQUÍ

print(formato_absa("batería", "La batería dura poco."))

In [ ]:
revisar(lambda: formato_absa("batería", "La batería dura poco.") == "[ASPECTO] batería [SEP] La batería dura poco.")

## Paso 5.3. Los datos (ficticios)

No hay un dataset público confiable de ABSA en español, así que usamos **300 reseñas ficticias** de relojes inteligentes, generadas combinando frases sobre cuatro aspectos (batería, pantalla, conexión y precio), cada una con dos aspectos y su sentimiento. Además, separamos **8 reseñas escritas a mano**, más naturales y con palabras distintas a las del entrenamiento, para probar si el modelo generaliza.

In [ ]:
FRASES = {
 "batería": {
   1: ["la batería dura casi dos días", "la batería aguanta todo el día sin problemas", "carga muy rápido y la batería rinde",
       "me sorprendió lo que dura la batería", "con el GPS activado la batería igual llega a la noche"],
   0: ["la batería no dura ni un día", "la batería se agota demasiado rápido", "hay que cargarlo todas las noches",
       "desde la actualización la batería dura la mitad", "la carga es lenta y la batería se acaba enseguida"]},
 "pantalla": {
   1: ["la pantalla se ve perfecta al sol", "la pantalla es brillante y nítida", "los colores de la pantalla son preciosos",
       "la pantalla es grande y fácil de leer", "la pantalla siempre encendida es muy útil"],
   0: ["la pantalla se rayó a la semana", "la pantalla se ve oscura en exteriores", "la pantalla es muy pequeña para leer",
       "la pantalla se apaga sola", "llegó con un pixel muerto en la pantalla"]},
 "conexión": {
   1: ["se conecta con el celular sin problemas", "la conexión bluetooth es estable", "sincroniza al instante con la aplicación",
       "las notificaciones del celular llegan al tiro", "nunca se ha desconectado del teléfono"],
   0: ["se desconecta del celular varias veces al día", "la aplicación no sincroniza los datos", "pierde la conexión bluetooth constantemente",
       "no reconoce mi teléfono después de la actualización", "las notificaciones llegan tarde o no llegan"]},
 "precio": {
   1: ["el precio es muy conveniente", "lo compré en oferta y vale cada peso", "excelente relación precio calidad",
       "es barato para todo lo que trae", "por ese precio no hay nada mejor"],
   0: ["es demasiado caro para lo que ofrece", "el precio es exagerado", "por ese valor esperaba mucho más",
       "hay modelos más baratos que hacen lo mismo", "no vale lo que cuesta"]},
}
CONECTORES = {(1, 1): [", además ", " y ", ". También "], (0, 0): [", además ", " y encima ", ". Para colmo "],
              (1, 0): [", pero ", " aunque ", ", sin embargo "], (0, 1): [", pero ", " aunque ", ", eso sí, "]}
PRUEBA_REAL = [
 ("Muy cómodo y la pantalla se ve increíble, lástima que la batería no alcanza para el día completo.", {"pantalla": 1, "batería": 0}),
 ("Lo pagué carísimo y además se desconecta del teléfono a cada rato.", {"precio": 0, "conexión": 0}),
 ("Batería de lujo, dura tres días. La app a veces no sincroniza el sueño.", {"batería": 1, "conexión": 0}),
 ("Por lo que cuesta es una ganga, y la pantalla es enorme.", {"precio": 1, "pantalla": 1}),
 ("La pantalla se raya con nada, pero la carga dura bastante.", {"pantalla": 0, "batería": 1}),
 ("Conecta rapidísimo con mi celular; eso sí, no lo encuentro barato.", {"conexión": 1, "precio": 0}),
 ("Se le acaba la carga en medio día y la pantalla queda negra de repente.", {"batería": 0, "pantalla": 0}),
 ("Las notificaciones llegan perfecto y vale cada peso que pagué.", {"conexión": 1, "precio": 1}),
]
def generar(n=300, semilla=0):
    import random
    rng = random.Random(semilla); filas = []
    aspectos = list(FRASES)
    for i in range(n):
        a1, a2 = rng.sample(aspectos, 2)
        s1, s2 = rng.randint(0, 1), rng.randint(0, 1)
        f1, f2 = rng.choice(FRASES[a1][s1]), rng.choice(FRASES[a2][s2])
        texto = f1[0].upper() + f1[1:] + rng.choice(CONECTORES[(s1, s2)]) + f2 + "."
        filas.append((i, texto, a1, s1)); filas.append((i, texto, a2, s2))
    return filas

filas = generar(300, semilla=0)
absa = pd.DataFrame(filas, columns=["resena", "texto", "aspecto", "label"])
absa["entrada"] = [formato_absa(a, t) for a, t in zip(absa["aspecto"], absa["texto"])]

# Separamos por reseña (no por fila), para que las dos filas de una reseña queden en el mismo conjunto
ids_val = set(absa["resena"].drop_duplicates().sample(frac=0.2, random_state=0))
absa_tr, absa_va = absa[~absa["resena"].isin(ids_val)], absa[absa["resena"].isin(ids_val)]
print("Filas de entrenamiento:", len(absa_tr), "| validación:", len(absa_va))
absa[["entrada", "label"]].head(4)

**Por qué separar por reseña:** cada reseña genera dos filas que comparten el texto. Si una fila queda en entrenamiento y la otra en validación, el modelo ya "vio" el texto: la validación sería optimista. Es la fuga de datos de la Clase 4.

## Paso 5.4. Ajustar con el token especial `[ASPECTO]` (láminas 121 a 123)

`[SEP]` ya existe en el tokenizador de BERT, pero `[ASPECTO]` no: lo agregamos como **token especial**, y agrandamos la tabla de embeddings del modelo (Clase 13) para que tenga una fila para él.

In [ ]:
MODELO_ABSA = "dccuchile/bert-base-spanish-wwm-uncased"
tok_absa = AutoTokenizer.from_pretrained(MODELO_ABSA)
tok_absa.add_special_tokens({"additional_special_tokens": ["[ASPECTO]"]})

modelo_absa = AutoModelForSequenceClassification.from_pretrained(MODELO_ABSA, **ETIQUETAS)
modelo_absa.resize_token_embeddings(len(tok_absa))           # una fila nueva en la tabla de embeddings

print(tok_absa.tokenize(absa["entrada"].iloc[0])[:12])

def tok_fn(lote):
    return tok_absa(lote["entrada"], padding="max_length", truncation=True, max_length=64)

dsa_tr = Dataset.from_pandas(absa_tr[["entrada", "label"]].reset_index(drop=True)).map(tok_fn, batched=True)
dsa_va = Dataset.from_pandas(absa_va[["entrada", "label"]].reset_index(drop=True)).map(tok_fn, batched=True)

trainer_absa = Trainer(model=modelo_absa, args=argumentos("./resultados_absa", 3e-5, epocas=3),
                       train_dataset=dsa_tr, eval_dataset=dsa_va, compute_metrics=metricas)
trainer_absa.train()

**Qué deberías ver:** `[ASPECTO]` como un solo token (no partido en pedazos) y una exactitud de validación muy alta, cercana a 1: las reseñas ficticias son fáciles, porque usan siempre las mismas frases. La prueba de verdad son las reseñas escritas a mano.

## Paso 5.5. Evaluar por aspecto (láminas 124 a 128)

In [ ]:
pred_va = trainer_absa.predict(dsa_va).predictions.argmax(axis=-1)
eval_aspecto = absa_va.reset_index(drop=True).assign(pred=pred_va)

por_aspecto = pd.DataFrame([{"aspecto": a,
                             "exactitud": accuracy_score(g["label"], g["pred"]),
                             "f1": f1_score(g["label"], g["pred"], zero_division=0)}
                            for a, g in eval_aspecto.groupby("aspecto")]).set_index("aspecto")
por_aspecto.plot(kind="bar", figsize=(7, 3.5), ylim=(0, 1.05), rot=0, title="Métricas por aspecto (validación)")
plt.show()
por_aspecto.round(3)

### Ejercicio 10
**La prueba de verdad.** Clasifica las 8 reseñas escritas a mano (`PRUEBA_REAL`): para cada reseña y cada aspecto de su diccionario, arma la entrada con `formato_absa`, predice con `clasif_absa` y compara con la etiqueta. Guarda la exactitud en `exac_mano`.

In [ ]:
clasif_absa = pipeline("text-classification", model=trainer_absa.model, tokenizer=tok_absa, device=DISPOSITIVO)

exac_mano = None  # TU CÓDIGO AQUÍ
print(exac_mano)

In [ ]:
_ent = [(formato_absa(a, t), s) for t, d in PRUEBA_REAL for a, s in d.items()]
_pred = [1 if r["label"] == "positiva" else 0 for r in clasif_absa([e for e, _ in _ent])]
revisar(lambda: np.isclose(exac_mano, accuracy_score([s for _, s in _ent], _pred)))

**Qué deberías ver:** una exactitud **menor** que en validación (entre 0,6 y 0,9), porque las reseñas a mano usan palabras que el modelo no vio ("carísimo", "ganga", "queda negra"). Es la lección de siempre: **los datos de entrenamiento definen lo que el modelo sabe**. Con datos ficticios y repetitivos, el modelo aprende las frases, no la idea.

## Paso 5.6. ABSA sin entrenar: zero-shot

Una alternativa sin datos etiquetados: el modelo de **zero-shot** de la Clase 17, preguntándole por cada aspecto.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=DISPOSITIVO)

def absa_zero_shot(texto, aspecto):
    r = zero_shot(texto, candidate_labels=["positiva", "negativa"],
                  hypothesis_template=f"La opinión sobre {aspecto} es {{}}.")
    return 1 if r["labels"][0] == "positiva" else 0

reales, pred_zs = [], []
for texto, aspectos in PRUEBA_REAL:
    for aspecto, sentimiento in aspectos.items():
        reales.append(sentimiento); pred_zs.append(absa_zero_shot(texto, aspecto))
exac_zero_shot = accuracy_score(reales, pred_zs)
print(f"Reseñas a mano | ajustado con datos ficticios: {exac_mano} | zero-shot: {exac_zero_shot:.3f}")

**Qué deberías ver:** el zero-shot compitiendo con el modelo ajustado, o incluso ganándole, **sin ningún dato de entrenamiento**. En un proyecto real, el camino suele ser: partir con zero-shot, etiquetar algunos cientos de reseñas **reales** y, con ellas, ajustar.

### Ejercicio 11
Usa `absa_zero_shot` con tu propia reseña de un producto (en `resena_mixta`), que sea **positiva sobre un aspecto y negativa sobre otro**, y guarda en `mis_aspectos` un diccionario `{aspecto: predicción}` para esos dos aspectos.

In [ ]:
resena_mixta = ""      # TU RESEÑA AQUÍ
mis_aspectos = None    # TU CÓDIGO AQUÍ, por ejemplo {"batería": absa_zero_shot(...), "precio": ...}
print(mis_aspectos)

In [ ]:
revisar(lambda: len(resena_mixta.split()) >= 5 and isinstance(mis_aspectos, dict) and len(mis_aspectos) == 2
        and all(v == absa_zero_shot(resena_mixta, k) for k, v in mis_aspectos.items()))

---
# Parte 6. Cierre del curso de contenidos

## Todo lo que vimos

| Capítulo | Clases | Herramienta | Cuándo usarla en el capstone |
|---|---|---|---|
| 5. Aprendizaje supervisado | 2 a 6 | Regresión, árboles, random forest, métricas | **Datos en tabla con una variable a predecir.** Siempre como primera opción y como línea base |
| 6. No supervisado | 7 a 10 | K-medias, jerárquico, DBSCAN, PCA | Segmentar, encontrar grupos o casos raros sin etiquetas |
| 7. Texto | 11 a 13 | Regex, TF-IDF, temas, n-gramas | Primera aproximación a cualquier problema con texto |
| 8. Redes neuronales | 14 y 15 | Keras: densas, CNN, RNN | Imágenes; series de tiempo con muchos datos |
| 9. Temas de deep learning | 16 y 17 | CNN y LSTM a fondo, autoencoders | Detectar anomalías; mejorar CNN |
| 10. LLM | 17 y 18 | Hugging Face, fine-tuning, LoRA, cuantización | **Texto: partir con pipeline o zero-shot, y ajustar si hay datos etiquetados** |

## Cinco reglas para el capstone

1. **Partir por lo simple** y compararlo todo contra una línea base.
2. **Evaluar con datos que el modelo no vio**, y decidir con validación, no con prueba.
3. **Elegir la métrica según el problema** (Clase 7): exactitud no basta si hay desbalance.
4. **Los datos definen el techo**: mejorar los datos suele rendir más que complicar el modelo.
5. **Reportar con honestidad**, incluyendo lo que no funcionó.

### Ejercicio 12
Escribe en la celda siguiente qué herramienta de este curso usarás primero en tu proyecto capstone y por qué. (No tiene verificación: compártanlo con su equipo.)

**Mi respuesta:** (doble clic para escribir)

Guarden el notebook en `ejercicios_estudiantes` como `clase18_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
La media de una columna de 0 y 1 es la proporción de unos.

In [ ]:
prop_positivas = entreno["label"].mean()
print(prop_positivas)

### Ejercicio 2
El primer carácter de `"4 stars"` es `"4"`; se convierte a entero y se compara.

In [ ]:
pred_sin_ajustar = [1 if int(s["label"][0]) >= 4 else 0 for s in salidas_estrellas]
exac_sin_ajustar = accuracy_score(prueba["label"], pred_sin_ajustar)
resultados["Preentrenado sin ajustar (Clase 17)"] = exac_sin_ajustar
print(round(exac_sin_ajustar, 3))

### Ejercicio 3

In [ ]:
exac_ft = accuracy_score(prueba["label"], pred_ft)
resultados["Fine-tuning completo"] = exac_ft
pd.Series(resultados, name="Exactitud en prueba").sort_values(ascending=False).round(3)

### Ejercicio 4
Cualquier reseña de 3 o más palabras.

In [ ]:
mi_resena = "La calidad es pésima, no lo vuelvo a comprar."
etiqueta_mi_resena = mi_clasificador(mi_resena)[0]["label"]
print(etiqueta_mi_resena)

### Ejercicio 5
A tiene 16 × 768 y B tiene 768 × 16: 12.288 + 12.288 = 24.576, el doble que con r = 8.

In [ ]:
params_lora_r16 = 16 * 768 + 768 * 16
print(params_lora_r16)

### Ejercicio 6

In [ ]:
pct_entrenable = entrenables / total_params * 100
print(round(pct_entrenable, 3), "%")

### Ejercicio 7

In [ ]:
reduccion_pct = (mb_fp32 - mb_int8) / mb_fp32 * 100
print(round(reduccion_pct, 1), "%")

### Ejercicio 8

In [ ]:
aceleracion = t_32 / t_8
perdida_exactitud = exac_32 - exac_8
print(round(aceleracion, 2), "veces más rápido | pérdida:", round(perdida_exactitud, 3))

### Ejercicio 9

In [ ]:
def formato_absa(aspecto, texto):
    return f"[ASPECTO] {aspecto} [SEP] {texto}"

print(formato_absa("batería", "La batería dura poco."))

### Ejercicio 10
Se arma una entrada por cada par (reseña, aspecto): 16 en total.

In [ ]:
entradas_mano, reales_mano = [], []
for texto, aspectos in PRUEBA_REAL:
    for aspecto, sentimiento in aspectos.items():
        entradas_mano.append(formato_absa(aspecto, texto)); reales_mano.append(sentimiento)
pred_mano = [1 if r["label"] == "positiva" else 0 for r in clasif_absa(entradas_mano)]
exac_mano = accuracy_score(reales_mano, pred_mano)
print(round(exac_mano, 3))
for e, r, p in zip(entradas_mano, reales_mano, pred_mano):
    print("bien" if r == p else "MAL ", "|", e[:90])

### Ejercicio 11
Cualquier reseña mixta, por ejemplo:

In [ ]:
resena_mixta = "La cámara saca fotos espectaculares, pero el teléfono se calienta demasiado."
mis_aspectos = {"cámara": absa_zero_shot(resena_mixta, "la cámara"), "temperatura": absa_zero_shot(resena_mixta, "la temperatura")}
print(mis_aspectos)

### Ejercicio 12
Depende de cada proyecto. Lo importante es que la elección se justifique con el tipo de datos y que haya una línea base.